# News Articles Grouping Research: Baseline

In this notebook, we develop a baseline grouping pipeline based on dense embeddings and the HDBSCAN clustering algorithm.

The objective of this research is to automatically group news articles into meaningful real-world events. In other words, we do not want to cluster articles together simply because they mention the same high-level entity (e.g., the President of Argentina). Instead, we aim to group them according to the specific event that occurred. For example:

- *The Argentine president approves a set of laws preventing X, Y, and Z.*
- *The Argentine president vetoes a tax reform proposal.*

Although both articles mention the same public figure, they describe different real-world events and therefore should belong to different clusters.

This baseline consists of a single clustering layer (HDBSCAN) applied on top of the embedding representation. Its purpose is to establish a reference point for the research, allowing us to identify the limitations of purely semantic clustering and determine where additional modeling layers are required to achieve more accurate event-level grouping.

<br/>

## Experiment Structure

### 1. Stage 1 - Generate Embeddings

Using `multilingual-e5-large`, we generate fixed-size vector representations (1024 dimensions) for each article based on the concatenation of its title and content.

The dataset and embeddings remain constant across experiments. Embeddings are computed once and stored, then reused in subsequent stages to ensure consistency and computational efficiency.

In addition, we apply **Principal Component Analysis (PCA)** to reduce the dimensionality of the embeddings to 100.  

Dimensionality reduction serves two main purposes:

- It can help remove redundant or low-variance components, potentially reducing noise in the representation.
- It improves computational efficiency, allowing HDBSCAN to run faster and scale more effectively.

By projecting the embeddings into a lower-dimensional space, density structures may become more compact and easier for the clustering algorithm to identify. However, the number of retained components must be chosen carefully to avoid discarding information that is important for distinguishing between events.

### 2. Stage 2 - HDBSCAN Clustering

We apply **HDBSCAN (Hierarchical Density-Based Spatial Clustering)**, a density-based clustering algorithm that:

- Does not require specifying the number of clusters in advance.
- Can identify clusters of varying density.
- Explicitly labels outliers as noise.

HDBSCAN groups articles based on embedding proximity in the vector space. While effective in capturing semantic similarity, this approach may fail to distinguish between different real-world events that share overlapping entities or general topics. This limitation motivates the need for a more structured, multi-layered pipeline in subsequent experiments.

> **Note:** Since the number of articles to cluster is very large, we use a **CUDA-accelerated implementation of HDBSCAN** to leverage the GPU available in our environment and significantly speed up the clustering process.

## Import Dependencies

In [ ]:
!pip install cudf-cu12 cuml-cu12 --extra-index-url=https://pypi.nvidia.com

In [2]:
import numpy as np
import pandas as pd
import cupy as cp

from cuml.cluster import HDBSCAN as cuHDBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import homogeneity_completeness_v_measure
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
DATASET_PATH = "/content/drive/MyDrive/Research/articles_grouping/datasets/preprocessed/main/articles_events_2500.csv"
EMBEDDINGS_PATH = "/content/drive/MyDrive/Research/articles_grouping/datasets/preprocessed/main/embeddings_2500.dat"

## 1. Data Preparation

In [4]:
# Setup Google Drive in Colab
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### Load Dataset

In [5]:
df = pd.read_csv(filepath_or_buffer=DATASET_PATH)
df = df.reset_index(drop=True)

print(f"Dataset loaded and formatted successfully with {df.shape[0]} examples")

Dataset loaded and formatted successfully with 163753 examples


### Load Embeddings to Memory

In [6]:
TOTAL = len(df)
EMBEDDING_DIM = 1024

embeddings = np.fromfile(EMBEDDINGS_PATH, dtype="float32").reshape(TOTAL, EMBEDDING_DIM)

In [7]:
print(f"Embeddings loaded with shape: {embeddings.shape}")
print(f"Total number of articles (rows): {embeddings.shape[0]}")
print(f"Embedding dimension (columns): {embeddings.shape[1]}")

Embeddings loaded with shape: (163753, 1024)
Total number of articles (rows): 163753
Embedding dimension (columns): 1024


### Principal Component Analysis

In [8]:
N_COMPONENTS = 100
pca = PCA(n_components=N_COMPONENTS)
pca.fit(embeddings)

embeddings_pca = pca.transform(embeddings)

In [9]:
print(f"PCA embeddings shape: {embeddings_pca.shape}")
print(f"Total number of articles (rows): {embeddings_pca.shape[0]}")
print(f"PCA Embedding dimension (columns): {embeddings_pca.shape[1]}")

PCA embeddings shape: (163753, 100)
Total number of articles (rows): 163753
PCA Embedding dimension (columns): 100


## 2. HDBSCAN Clustering

In [10]:
# Initialize clustering algorithm
METRIC = "euclidean"
MIN_CLUSTER_SIZE = 10
MIN_SAMPLES = 3
CLUSTER_SELECTION_EPSILON = 0.1
ALPHA = 1
CLUSTER_SELECTION_METHOD = "leaf"

In [11]:
# Move embeddings to GPU memory
embeddings_gpu = cp.asarray(embeddings_pca)

# Initialize GPU HDBSCAN version
gpu_clusterer = cuHDBSCAN(metric=METRIC,
                          min_cluster_size=MIN_CLUSTER_SIZE,
                          min_samples=MIN_SAMPLES,
                          cluster_selection_epsilon=CLUSTER_SELECTION_EPSILON,
                          alpha=ALPHA,
                          cluster_selection_method=CLUSTER_SELECTION_METHOD)

print("Starting GPU-accelerated HDBSCAN clustering...")
cluster_labels_gpu = gpu_clusterer.fit_predict(embeddings_gpu)

# Move labels back to CPU/Numpy for consistency with existing code
cluster_labels = cp.asnumpy(cluster_labels_gpu)
print(f"Clustering complete. Found {len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)} clusters.")

Starting GPU-accelerated HDBSCAN clustering...
Clustering complete. Found 4588 clusters.


> *Note: Clustering the 163,753 articles into 4,582 clusters took 27 seconds.*

In [12]:
# Assign cluster id to each article in the dataframe
df["cluster_id"] = cluster_labels

### Cluster Evaluation

To assess the quality of the generated clusters, we evaluate them using two complementary perspectives:

1. **Global Event Consistency:**  
   This evaluates how well the clustering aligns with the ground-truth event labels in the dataset. Metrics such as **Homogeneity**, **Completeness**, and **V-Measure** are used to determine whether articles from the same event are grouped together and whether clusters contain articles from multiple events.

2. **Intra-cluster Similarity:**  
   This measures the average **cosine similarity** between article embeddings within the same cluster. The objective is to evaluate how semantically coherent each cluster is in the embedding space.

To avoid this distortion, each `-1` article was reassigned a unique negative integer identifier, effectively treating every unassigned article as its own singleton cluster. This ensures the evaluation reflects the true behavior of the pipeline, unassigned articles are isolated, not grouped together.

In [15]:
print(f"Amount of noise articles: {len(df[df["cluster_id"] == -1])}")
print(f"Percentage of noise articles: {(len(df[df["cluster_id"] == -1]) / len(df)) * 100:.1f}%")

Amount of noise articles: 63396
Percentage of noise articles: 38.7%


> **63,396** noise articles, which represent 38.7% of the total articles/

In [16]:
# Assign unique IDs to singletons
fixed_df = df["cluster_id"].copy()
noise_mask = fixed_df == -1
fixed_df[noise_mask] = range(-1, -noise_mask.sum() - 1, -1)

In [ ]:
y_true = df["event_id"]
y_pred = fixed_df

homogeneity, completeness, v_measure = homogeneity_completeness_v_measure(y_true, y_pred)

# Each cluster contains only one class
print(f"Homogeneity Score: {homogeneity:.4f}")
# All samples of a class are in the same cluster
print(f"Completeness Score: {completeness:.4f}")
# Harmonic mean of the two
print(f"V-Measure Score: {v_measure:.4f}")

In [ ]:
def calculate_intra_sim(df, embeddings, n_clusters_to_sample=4_000):
    valid_clusters = df[df['cluster_id'] != -1]['cluster_id'].unique()
    sampled_clusters = np.random.choice(valid_clusters, min(len(valid_clusters), n_clusters_to_sample), replace=False)

    similarities = []
    for cid in sampled_clusters:
        indices = df[df['cluster_id'] == cid].index
        if len(indices) > 1:
            cluster_embeds = embeddings[indices]
            sim_matrix = cosine_similarity(cluster_embeds)
            # Take upper triangle to avoid diagonal and double counting
            upper_sims = sim_matrix[np.triu_indices(len(indices), k=1)]
            similarities.append(np.mean(upper_sims))

    return np.mean(similarities) if similarities else 0

avg_sim = calculate_intra_sim(df, embeddings_pca)
print(f"\nAverage Intra-cluster Cosine Similarity (sampled): {avg_sim:.4f}")

### Results

After evaluating both global clustering metrics and intra-cluster similarity, we observe the following results:

- **Homogeneity Score:** 0.9898  
- **Completeness Score:** 0.7510  
- **V-Measure Score:** 0.8540  
- **Average Intra-cluster Cosine Similarity:** 0.8821  

A key observation is that **~38.7% of the articles (63,396)** were labeled as **noise** by HDBSCAN. This indicates that the algorithm prefers to sacrifice coverage in order to maintain safer and more reliable clusters. While this behavior helps prevent unrelated articles from being grouped together, it also means that a large portion of the dataset remains unclustered.

The results reveal an important characteristic of **HDBSCAN as an event clustering baseline**. The **homogeneity score of 0.99** shows that clusters are extremely pure: when HDBSCAN groups articles together, they almost always belong to the same ground-truth event. However, this high purity is partly explained by the large number of articles excluded as noise.

The **completeness score of 0.75** indicates that many true event connections are missed. Articles that belong to the same event are often left as isolated noise points or distributed across different clusters instead of being grouped together.

The **average intra-cluster cosine similarity of 0.882** further confirms that the clusters produced by HDBSCAN are semantically coherent. However, cosine similarity primarily captures topical similarity rather than precise event identity. As a result, HDBSCAN may struggle when different events share strong semantic overlap.

These limitations motivate the **multi-signal graph-based approach** proposed in this research, which aims to improve **completeness and overall V-Measure while maintaining high cluster purity**.

# Conclusion

Wrapping up the baseline experimentation, our goal was to establish a starting point for the main research. The objective is to design a pipeline composed of multiple stages that can group news articles into events with the level of precision and granularity required for real-world event detection.

Although some of the results obtained here are promising, and this approach would perform very well if the task were focused only on **topic-level grouping**, our research aims to go a step further and accurately detect **specific real-world events**.

In conclusion, this experiment provides a solid baseline and reference point from which we can analyze the limitations of purely semantic clustering and continue improving the pipeline in subsequent stages.